# EDA & Missing-Value Handling

Reads data from the `ml_features` view (`sql/views.sql`, already filtered to
`data_source='historical'`) — base columns only, no dashboard aggregates.

Checklist:
- [x] `loan_status` distribution (baseline: 21.8% default rate on historical data)
- [x] Systematic leakage scan across every categorical column (default-rate spread) and
      every numeric-categorical pair (within-group variance ratio) — not just the two
      columns already under suspicion
- [x] `loan_grade` vs. default correlation — if the A vs. G gap exceeds ~90%, document the leakage decision in the model section
- [x] `loan_int_rate` missing pattern: random, or does it follow `loan_grade`?
- [x] Confirm no `person_age` values > 100 remain after `clean_outliers()`
- [x] Compare distributions before/after imputation

In [1]:
import pandas as pd
from sqlalchemy import text
import sys
sys.path.insert(0, '../etl')
from config import get_engine

engine = get_engine()
df = pd.read_sql(text("SELECT * FROM ml_features"), engine)
df.head()

,loan_id,client_id,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,debt_to_income_ratio,loan_status,age,income,home_ownership,emp_length,default_on_file,cred_hist_length,credit_utilization_ratio,past_delinquencies,country
0,4,CUST_00001,PERSONAL,D,35000.0,16.02,0.59,0.7356,1,22,59000.0,RENT,4.0,Y,3,0.4956,0,Canada
1,5,CUST_00002,EDUCATION,B,1000.0,11.14,0.10,0.2716,0,21,9600.0,OWN,5.0,N,2,0.5854,3,Canada
2,6,CUST_00003,MEDICAL,C,5500.0,12.87,0.57,0.8605,1,25,9600.0,MORTGAGE,1.0,N,3,0.7507,0,UK
3,7,CUST_00004,MEDICAL,C,35000.0,15.23,0.53,0.6436,1,23,65500.0,RENT,4.0,N,2,0.3793,0,Canada
4,8,CUST_00005,MEDICAL,C,35000.0,14.27,0.55,0.9306,1,24,54400.0,RENT,8.0,Y,4,0.2281,0,USA


In [2]:
print("Rows, columns:", df.shape)
print("\nDtypes:\n", df.dtypes)
na_counts = df.isna().sum()
print("\nMissing values in the Postgres feature view:\n", na_counts[na_counts > 0] if na_counts.sum() else "none")

Rows, columns: (32581, 18)

Dtypes:
 loan_id                       int64
client_id                       str
loan_intent                     str
loan_grade                      str
loan_amnt                   float64
loan_int_rate               float64
loan_percent_income         float64
debt_to_income_ratio        float64
loan_status                   int64
age                           int64
income                      float64
home_ownership                  str
emp_length                  float64
default_on_file                 str
cred_hist_length              int64
credit_utilization_ratio    float64
past_delinquencies            int64
country                         str
dtype: object

Missing values in the Postgres feature view:
 none


## Why there's no missingness left in Postgres

`historical_load.py::impute_missing` already median-imputes `person_emp_length` and
`loan_int_rate` before the ETL inserts rows, so the feature view above has zero nulls.
To check whether the *original* missingness was random (MCAR) or followed a pattern
(MAR), the next cells go back to the raw source file, before imputation.

In [3]:
raw = pd.read_excel('../data/Credit_Risk_Dataset.xlsx')
na_raw = raw.isna().sum()
na_raw[na_raw > 0]

person_emp_length     895
loan_int_rate        3116
dtype: int64

In [4]:
import plotly.express as px

missing_rate_by_grade = (raw.groupby('loan_grade')['loan_int_rate']
                          .apply(lambda s: s.isna().mean() * 100)
                          .round(2)
                          .reset_index(name='pct_missing'))

fig_missing_rate = px.bar(
    missing_rate_by_grade, x='loan_grade', y='pct_missing',
    title='loan_int_rate missing rate by loan_grade (raw source data)',
    labels={'pct_missing': '% missing', 'loan_grade': 'Loan grade'},
)
fig_missing_rate.show()
missing_rate_by_grade

,loan_grade,pct_missing
0,A,9.31
1,B,10.10
2,C,9.76
3,D,8.60
4,E,8.61
5,F,11.20
6,G,7.81


In [5]:
missing_emp_by_type = (raw.groupby('employment_type')['person_emp_length']
                        .apply(lambda s: s.isna().mean() * 100)
                        .round(2)
                        .reset_index(name='pct_missing'))

fig_missing_emp = px.bar(
    missing_emp_by_type, x='employment_type', y='pct_missing',
    title='person_emp_length missing rate by employment_type (raw source data)',
    labels={'pct_missing': '% missing', 'employment_type': 'Employment type'},
)
fig_missing_emp.show()
missing_emp_by_type

,employment_type,pct_missing
0,Full-time,2.77
1,Part-time,2.80
2,Self-employed,2.78
3,Unemployed,2.21


**Finding**: `loan_int_rate` missingness ranges 7.8-11.2% across grades A-G, and
`person_emp_length` missingness ranges 2.2-2.8% across employment types, both without a
clear trend (`Unemployed` is not the highest, contrary to the intuitive guess). Neither
column's missingness tracks the variable most likely to explain it — consistent with
**MCAR (missing completely at random)** rather than a grade- or employment-driven
pattern. Median imputation (already applied in the ETL) is a reasonable choice here.

## `loan_status` distribution

In [6]:
status_pct = (df['loan_status'].value_counts(normalize=True) * 100).round(2)
status_df = pd.DataFrame({'loan_status': status_pct.index.astype(str), 'pct': status_pct.values})

fig_status = px.bar(
    status_df, x='loan_status', y='pct',
    title='loan_status distribution (historical)',
    labels={'pct': '% of applications', 'loan_status': 'loan_status (1 = default)'},
)
fig_status.show()
status_df

,loan_status,pct
0,0,78.18
1,1,21.82


## Systematic leakage scan (method, not just the two columns already under suspicion)

Two general techniques, applied to every column instead of just `loan_grade`/
`loan_int_rate`, so the method would surface a leaky column even without already
knowing to look for one:

1. **Per-category default rate spread**: for every categorical column, compute
   `loan_status` mean per group. A spread that stretches toward the 0%/100% extremes
   (near-total separation) is the leakage signal — a legitimate risk factor nudges the
   rate, it doesn't split it almost perfectly.
2. **Within-group variance ratio**: for every categorical column, check how much a
   numeric column's variance shrinks once grouped by it (`within-group variance /
   overall variance`). A ratio near 0 means the numeric column is close to
   deterministic given the category — the two are encoding the same information twice.

In [7]:
categorical_cols = ['loan_intent', 'loan_grade', 'home_ownership', 'default_on_file', 'country']

separation_scan = []
for col in categorical_cols:
    rates = df.groupby(col)['loan_status'].mean() * 100
    separation_scan.append({
        'column': col,
        'n_groups': rates.shape[0],
        'min_default_pct': round(rates.min(), 2),
        'max_default_pct': round(rates.max(), 2),
        'spread': round(rates.max() - rates.min(), 2),
    })

separation_df = pd.DataFrame(separation_scan).sort_values('spread', ascending=False).reset_index(drop=True)

fig_separation = px.bar(
    separation_df, x='column', y='spread',
    title='Default-rate spread (max - min) per categorical column',
    labels={'spread': 'spread in % default across groups'},
)
fig_separation.show()
separation_df

,column,n_groups,min_default_pct,max_default_pct,spread
0,loan_grade,7,9.96,98.44,88.48
1,home_ownership,4,7.47,31.57,24.10
2,default_on_file,2,18.39,37.81,19.41
3,loan_intent,6,14.81,28.59,13.78
4,country,3,21.73,21.86,0.13


In [8]:
numeric_cols = ['loan_amnt', 'loan_int_rate', 'loan_percent_income',
                'debt_to_income_ratio', 'income', 'credit_utilization_ratio']

variance_scan = []
for cat_col in categorical_cols:
    for num_col in numeric_cols:
        overall_var = df[num_col].var()
        within_group_var = df.groupby(cat_col)[num_col].var().mean()
        ratio = within_group_var / overall_var if overall_var else float('nan')
        variance_scan.append({
            'categorical': cat_col, 'numeric': num_col,
            'within_group_var_ratio': round(ratio, 4),
        })

variance_df = pd.DataFrame(variance_scan).sort_values('within_group_var_ratio').reset_index(drop=True)
variance_df.head(10)

,categorical,numeric,within_group_var_ratio
0,loan_grade,loan_int_rate,0.3930
1,default_on_file,loan_int_rate,0.6240
2,loan_grade,income,0.8994
3,home_ownership,loan_int_rate,0.9098
4,default_on_file,income,0.9405
5,home_ownership,loan_amnt,0.9668
6,loan_intent,income,0.9945
7,home_ownership,credit_utilization_ratio,0.9951
8,default_on_file,credit_utilization_ratio,0.9968
9,loan_intent,credit_utilization_ratio,0.9991


**Domain question for whatever column tops either scan**: is this value known *at the
moment* a customer submits an application, or does the system only produce it *after* a
decision process (underwriting/risk scoring)? That's not something the data itself can
answer — it has to be reasoned about column by column.

`loan_grade` tops the separation scan by a wide margin: an 88.5-point spread (9.96% →
98.44%), vs. 24.1 for the next-highest column (`home_ownership`) and near-zero for
`country` — an order of magnitude beyond anything a normal risk factor produces.
`loan_grade` → `loan_int_rate` also has the lowest within-group variance ratio (0.39,
vs. 0.62 for the next-lowest pair): grouping by grade cuts `loan_int_rate`'s variance by
more than half, again the clear outlier rather than a marginal one. Both `loan_grade`
and `loan_int_rate` are underwriting outputs, not application-time inputs — that's what
turns "statistically the biggest outlier" into "actually leakage." Detail on those two
specifically follows.

## `loan_grade` / `loan_int_rate` vs. `loan_status` — leakage check

In [9]:
default_by_grade = (df.groupby('loan_grade')['loan_status'].mean() * 100).round(2).sort_index()
default_by_grade = default_by_grade.reset_index(name='default_rate_pct')

fig_default_grade = px.bar(
    default_by_grade, x='loan_grade', y='default_rate_pct',
    title='Default rate by loan_grade',
    labels={'default_rate_pct': '% default', 'loan_grade': 'Loan grade'},
)
fig_default_grade.show()
default_by_grade

,loan_grade,default_rate_pct
0,A,9.96
1,B,16.28
2,C,20.73
3,D,59.05
4,E,64.42
5,F,70.54
6,G,98.44


In [10]:
fig_rate_by_grade = px.box(
    df, x='loan_grade', y='loan_int_rate',
    category_orders={'loan_grade': sorted(df['loan_grade'].unique())},
    title='loan_int_rate by loan_grade',
)
fig_rate_by_grade.show()

### Decision: exclude `loan_grade` and `loan_int_rate` from the at-application feature set

Default rate by grade: **A=9.96%, B=16.28%, C=20.73%, D=59.05%, E=64.42%, F=70.54%,
G=98.44%** — a near-monotonic climb with near-total separation between the low and high
grades, and `loan_int_rate` is tightly clustered per grade (near-deterministic given
grade, see the boxplot above). Both columns are *outputs* of an underwriting/risk-scoring
step, not information available at the moment a customer submits a new application —
using them as model input would be leakage: the model would effectively be reading the
answer off a column computed after the fact.

**Decision for session 9 onward**: the *at-application* model (used in the Streamlit
prediction tab) drops both columns and uses only raw customer + credit-bureau data. The
*portfolio* model (used for analyzing the existing book) may keep them, since it is not
making an approval decision.

## Cross-check: `person_age` / `person_emp_length` outliers

Already blocked at ETL time (`historical_load.py::clean_outliers`) — this just confirms
nothing slipped through into Postgres.

In [11]:
print("age > 100:", (df['age'] > 100).sum())
print("age range:", df['age'].min(), "-", df['age'].max())
print("emp_length > age - 14 violations:", (df['emp_length'] > (df['age'] - 14)).sum())

age > 100: 0
age range: 20 - 94
emp_length > age - 14 violations: 0


## Distributions before vs. after median imputation

Raw = source file with missing values dropped. Postgres = after
`historical_load.py::impute_missing` filled them with the column median.

In [12]:
import plotly.graph_objects as go

fig_rate_compare = go.Figure()
fig_rate_compare.add_trace(go.Histogram(
    x=raw['loan_int_rate'].dropna(), name='Raw (pre-impute)',
    opacity=0.6, histnorm='probability density',
))
fig_rate_compare.add_trace(go.Histogram(
    x=df['loan_int_rate'], name='Postgres (post-impute)',
    opacity=0.6, histnorm='probability density',
))
fig_rate_compare.update_layout(
    barmode='overlay', title='loan_int_rate: before vs. after median imputation',
)
fig_rate_compare.show()

In [13]:
fig_emp_compare = go.Figure()
fig_emp_compare.add_trace(go.Histogram(
    x=raw['person_emp_length'].dropna(), name='Raw (pre-impute)',
    opacity=0.6, histnorm='probability density',
))
fig_emp_compare.add_trace(go.Histogram(
    x=df['emp_length'], name='Postgres (post-impute)',
    opacity=0.6, histnorm='probability density',
))
fig_emp_compare.update_layout(
    barmode='overlay', title='person_emp_length: before vs. after median imputation',
)
fig_emp_compare.show()

## Save key figures for the README

In [14]:
import os

os.makedirs('figures', exist_ok=True)
fig_status.write_image('figures/loan_status_distribution.png', scale=2)
fig_default_grade.write_image('figures/default_rate_by_grade.png', scale=2)
fig_rate_by_grade.write_image('figures/loan_int_rate_by_grade.png', scale=2)
print("Saved figures to notebooks/figures/")

Saved figures to notebooks/figures/
